In [1]:
import pandas as pd

df = pd.read_csv('Train.csv')
print(df.shape)
print(df.head())
print(df['Reached.on.Time_Y.N'].value_counts())
print(df.isna().sum())

(10999, 12)
   ID Warehouse_block Mode_of_Shipment  Customer_care_calls  Customer_rating  \
0   1               D           Flight                    4                2   
1   2               F           Flight                    4                5   
2   3               A           Flight                    2                2   
3   4               B           Flight                    3                3   
4   5               C           Flight                    2                2   

   Cost_of_the_Product  Prior_purchases Product_importance Gender  \
0                  177                3                low      F   
1                  216                2                low      M   
2                  183                4                low      M   
3                  176                4             medium      M   
4                  184                3             medium      F   

   Discount_offered  Weight_in_gms  Reached.on.Time_Y.N  
0                44           1233

In [2]:
report = {}
report['строк на входе'] = len(df)
report['дубликатов'] = df.duplicated().sum()
report['пропусков всего'] = int(df.isna().sum().sum())
report['доля доставлено вовремя'] = round((df['Reached.on.Time_Y.N'] == 1).mean(), 3)

print(report)

{'строк на входе': 10999, 'дубликатов': 0, 'пропусков всего': 0, 'доля доставлено вовремя': 0.597}


In [3]:
print('Всего заказов:', len(df))
print('Доставлено вовремя:', (df['Reached.on.Time_Y.N'] == 1).sum())
print('С опозданием', (df['Reached.on.Time_Y.N'] == 0).sum())
print('Средняя стоимость заказа', round(df['Cost_of_the_Product'].mean(), 1))
print('Средний вес, г:', round(df['Weight_in_gms'].mean(), 0))
print('Средняя скидка, %:', round(df['Discount_offered'].mean(), 1))
print('Средняя оценка клента:', round(df['Customer_rating'].mean(), 2))

Всего заказов: 10999
Доставлено вовремя: 6563
С опозданием 4436
Средняя стоимость заказа 210.2
Средний вес, г: 3634.0
Средняя скидка, %: 13.4
Средняя оценка клента: 2.99


In [4]:
mode_of_shipment = df.groupby('Mode_of_Shipment').agg(
    заказов=('ID', 'count'),
    доля_вовремя=('Reached.on.Time_Y.N', 'mean')
).round(3)
print(mode_of_shipment)

                  заказов  доля_вовремя
Mode_of_Shipment                       
Flight               1777         0.602
Road                 1760         0.588
Ship                 7462         0.598


способ доставки не связан с пунктуальностью; перевод заказов на авиа не даст улучшения

In [5]:
warehouse_block = df.groupby('Warehouse_block').agg(
    заказов=('ID', 'count'),
    доля_вовремя=('Reached.on.Time_Y.N', 'mean')
).round(3)
print(warehouse_block)

                 заказов  доля_вовремя
Warehouse_block                       
A                   1833         0.586
B                   1833         0.602
C                   1833         0.597
D                   1834         0.598
F                   3666         0.598


склад отправки не влияет на долю вовремя; узкое место — не в распределении по складам

In [6]:
bins = [0, 2000, 4000, 6000, 10000]
labels = ['0-2 кг', '2-4 кг', '4-6 кг', '6+ кг']
df['weight_bucket'] = pd.cut(df['Weight_in_gms'], bins=bins, labels=labels)

weight_bucket = df.groupby('weight_bucket', observed=True).agg(
    заказов=('ID', 'count'),
    доля_вовремя=('Reached.on.Time_Y.N', 'mean'),
    количество_опозданий=('Reached.on.Time_Y.N', lambda s: (1 - s.mean()) * s.count())
).round(3)

print(weight_bucket)

               заказов  доля_вовремя  количество_опозданий
weight_bucket                                             
0-2 кг            3245         0.678                1046.0
2-4 кг            1788         0.999                   1.0
4-6 кг            5958         0.431                3389.0
6+ кг                8         1.000                   0.0


Доля опозданий сильно зависит от веса посылки. В диапазоне 4–6 кг (5958 заказов, 54% всех) опаздывают 56,9% доставок — это главная зона потерь. В диапазоне 2–4 кг доходит вовремя 99,9% (всего 1 опоздание из 1788), при весе до 2 кг опаздывают 32,2%. Бакет 6+ кг (100% вовремя) неинформативен — там всего 8 заказов, вывод делать нельзя.

In [14]:
df['discount_bucket'] = pd.cut(df['Discount_offered'], bins=[0, 10, 30, 100], labels=['до 10%', '10-30%', 'больше 30%'])

discount_bucket = df.groupby('discount_bucket', observed=True).agg(
    заказов=('ID', 'count'),
    доля_вовремя=('Reached.on.Time_Y.N', 'mean'),
    количество_опозданий=('Reached.on.Time_Y.N', lambda s: (1 - s.mean()) * s.count())
).round(3)

print(discount_bucket)

                 заказов  доля_вовремя  количество_опозданий
discount_bucket                                             
до 10%              8352         0.469                4436.0
10-30%               937         1.000                   0.0
больше 30%          1710         1.000                   0.0


In [15]:
pd.crosstab(df['weight_bucket'], df['discount_bucket'], normalize='index').round(3)


discount_bucket,до 10%,10-30%,больше 30%
weight_bucket,,,
0-2 кг,0.654,0.123,0.223
2-4 кг,0.150,0.300,0.550
4-6 кг,1.000,0.000,0.000
6+ кг,0.250,0.250,0.500


Опоздания (40,3% всех заказов) сосредоточены в одном сегменте: посылки весом 4–6 кг со скидкой до 10% — там доходит вовремя лишь ~43%. Сегмент «2–4 кг, скидка от 10%» доезжает практически без потерь. Склад, способ доставки, пол и рейтинг клиента на пунктуальность не влияют. Число звонков в поддержку коррелирует с опозданиями (65% вовремя при 2 звонках против 52% при 6–7), но это следствие задержек, а не их причина — звонки можно использовать как индикатор проблемного заказа. Причинно-следственные связи о скидках по этим данным установить нельзя — есть только наблюдаемая закономерность.

In [8]:
calls = df.groupby('Customer_care_calls').agg(
    заказов=('ID', 'count'),
    доля_вовремя=('Reached.on.Time_Y.N', 'mean'),
    средняя_оценка=('Customer_rating', 'mean')
).round(3)

print(calls)

                     заказов  доля_вовремя  средняя_оценка
Customer_care_calls                                       
2                        638         0.652           3.017
3                       3217         0.625           2.960
4                       3557         0.598           2.998
5                       2328         0.584           2.973
6                       1013         0.516           3.084
7                        246         0.516           2.996


Клиенты с опоздавшими заказами чаше обращаются в поддержку...

In [9]:
pivot = df.pivot_table(
    index='Warehouse_block',
    columns='Mode_of_Shipment',
    values='Reached.on.Time_Y.N',
    aggfunc='mean'
).round(3)

print(pivot)

Mode_of_Shipment  Flight   Road   Ship
Warehouse_block                       
A                  0.586  0.571  0.590
B                  0.598  0.585  0.607
C                  0.624  0.575  0.596
D                  0.599  0.596  0.598
F                  0.601  0.601  0.597


In [13]:
kpi = pd.Series({
    'всего заказов': len(df),
    'доставлено вовремя': int((df['Reached.on.Time_Y.N'] == 1).sum()),
    'с опозданием': int((df['Reached.on.Time_Y.N'] == 0).sum()),
    'средняя стоимость заказа': round(df['Cost_of_the_Product'].mean(), 1),
    'средняя скидка, %': round(df['Discount_offered'].mean(), 1),
})

with pd.ExcelWriter('delivery_report.xlsx', engine='openpyxl') as writer:
    kpi.to_frame('значение').to_excel(writer, sheet_name='KPI')
    mode_of_shipment.to_excel(writer, sheet_name='Способ доставки')
    weight_bucket.to_excel(writer, sheet_name='Вес')
    discount_bucket.to_excel(writer, sheet_name='Скидка')
    pd.Series(report).to_frame('значение').to_excel(writer, sheet_name='Протокол')
    df.to_excel(writer, sheet_name='Данные', index=False)

